# Data Analysis of the filtered Materials dataset

---

In [1]:
import duckdb
import os

In [2]:
# Setup paths
output_dir = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\filtered"
filtered_materials_file = os.path.join(output_dir, "filtered_materials_encoded_time_cut.parquet")

In [3]:
con = duckdb.connect()

In [4]:
# Attach the parquet file as a table
con.execute(f"CREATE OR REPLACE TABLE materials AS SELECT * FROM '{filtered_materials_file}';")

### Missing percentage per column

In [5]:
print("Null percentage per column:")

# Get list of all column names using DESCRIBE
columns = con.execute(f"DESCRIBE SELECT * FROM '{filtered_materials_file}'").fetchdf()['column_name'].tolist()

results = []
total_rows = con.execute(f"SELECT COUNT(*) FROM '{filtered_materials_file}'").fetchone()[0]

for col in columns:
    null_count = con.execute(f"SELECT COUNT(*) - COUNT({col}) FROM '{filtered_materials_file}'").fetchone()[0]
    null_percentage = round(100.0 * null_count / total_rows, 2)
    results.append((col, null_percentage))

# Sort and print
results.sort(key=lambda x: x[1], reverse=True)
for col, perc in results:
    print(f"{col}: {perc:.2f}% nulls")

Null percentage per column:
component_position: 0.00% nulls
component_id: 0.00% nulls
serial_number_id: 0.00% nulls
station_id: 0.00% nulls
supplier_id: 0.00% nulls
mounting_place: 0.00% nulls
panel_position: 0.00% nulls
created_at: 0.00% nulls
book_state: 0.00% nulls
container_number_freq: 0.00% nulls


### Unique value count per column (Cardinality)

In [6]:
# Get column names
columns = con.execute("SELECT name FROM pragma_table_info('materials');").fetchdf()['name'].tolist()

# Calculate unique counts per column
results = []
for col in columns:
    unique_count = con.execute(f"SELECT COUNT(DISTINCT {col}) FROM materials;").fetchone()[0]
    results.append((col, unique_count))

# Print nicely
import pandas as pd
unique_counts = pd.DataFrame(results, columns=['column_name', 'unique_values'])
print("\n=== Unique Values per Column ===")
print(unique_counts)

categorical_cols = con.execute("""
    SELECT name
    FROM pragma_table_info('materials')
    WHERE type IN ('VARCHAR', 'STRING', 'TEXT');
""").fetchdf()['name'].tolist()

for col in categorical_cols:
    print(f"\n=== Top 10 Frequent Values for '{col}' ===")
    result = con.execute(f"""
        SELECT {col} AS value, COUNT(*) AS freq
        FROM materials
        GROUP BY {col}
        ORDER BY freq DESC
        LIMIT 10;
    """).fetchdf()
    print(result)


=== Unique Values per Column ===
             column_name  unique_values
0     component_position            429
1           component_id            166
2       serial_number_id          33223
3             station_id              3
4            supplier_id             31
5         mounting_place            628
6         panel_position              2
7             created_at          41008
8             book_state              2
9  container_number_freq           2228

=== Top 10 Frequent Values for 'component_position' ===
      value    freq
0  525f6012  157476
1  618dca23  142901
2  eb5ad6a6  139490
3  36e05322  135040
4  4055d7e5  118934
5  700225ab  115880
6  b066cba3  111092
7  c58d2ce3  111064
8  7529b1d3  109312
9  acc7c77e  108110

=== Top 10 Frequent Values for 'component_id' ===
      value    freq
0  e3c0f48d  343930
1  1a260d39  255977
2  8ddd5615  198439
3  198cd6b0  184715
4  e1f67b78  173696
5  93cf70a7  161587
6  eb51fc0b  146378
7  572738de  146131
8  332418ab  13984

### Dataset summary: number of rows, memory usage (approximate)

In [7]:
# Get total number of rows
summary = con.execute("""
    SELECT COUNT(*) AS total_rows
    FROM materials;
""").fetchdf()

print("\n=== 🗂️ Dataset Summary ===")
print(summary)

# Approximate size: Sum LENGTH for string columns
string_cols = con.execute("""
    SELECT name
    FROM pragma_table_info('materials')
    WHERE type IN ('VARCHAR', 'STRING', 'TEXT');
""").fetchdf()['name'].tolist()

if string_cols:
    length_sum_expr = " + ".join([f"LENGTH({col})" for col in string_cols])
    size_query = f"""
        SELECT ROUND(SUM({length_sum_expr}) / 1024 / 1024, 2) AS approx_string_data_MB
        FROM materials;
    """
    approx_size = con.execute(size_query).fetchdf()
    print("\n=== 📦 Approximate Size of String Data (MB) ===")
    print(approx_size)
else:
    print("\nNo string columns to estimate size.")

# Schema info
schema = con.execute("""
    SELECT name AS column_name, type AS data_type
    FROM pragma_table_info('materials');
""").fetchdf()
print("\n=== 📄 Schema Information ===")
print(schema)


=== 🗂️ Dataset Summary ===
   total_rows
0    10901764

=== 📦 Approximate Size of String Data (MB) ===
   approx_string_data_MB
0                 582.05

=== 📄 Schema Information ===
             column_name                 data_type
0     component_position                   VARCHAR
1           component_id                   VARCHAR
2       serial_number_id                   VARCHAR
3             station_id                   VARCHAR
4            supplier_id                   VARCHAR
5         mounting_place                   VARCHAR
6         panel_position                   VARCHAR
7             created_at  TIMESTAMP WITH TIME ZONE
8             book_state                   INTEGER
9  container_number_freq                    BIGINT


In [8]:
con.close()